In [ ]:
!apt-get update
!apt-get install -y chromium-browser
!apt-get install -y chromium-chromedriver

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. הגדרת הנתיב המדויק שנתת לי
file_path = '/content/warning-letters.xlsx'

print("מתחיל טעינת קובץ אקסל... זה עשוי לקחת דקה בגלל גודל הקובץ")

try:
    # 2. טעינת הנתונים
    df = pd.read_excel(file_path)
    print(f"הצלחנו! נטענו {len(df)} שורות מהאקסל.")

    # 3. ניקוי בסיסי ותאריכים
    date_col = 'Issue Date' if 'Issue Date' in df.columns else 'Posted Date'
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df['Year'] = df[date_col].dt.year

        # גרף מגמות לאורך שנים
        plt.figure(figsize=(10, 5))
        df['Year'].value_counts().sort_index().plot(kind='bar', color='skyblue')
        plt.title('FDA Warning Letters per Year')
        plt.show()

    # 4. שלב ה-TF-IDF (שליפת מילות מפתח מה-Subject)
    # ננקה ערכים ריקים כדי שהמודל לא יקרוס
    subjects = df['Subject'].fillna('Missing').astype(str)

    # הגדרת המודל - מחפש את 25 המילים הכי משמעותיות
    tfidf = TfidfVectorizer(stop_words='english', max_features=25)
    tfidf_matrix = tfidf.fit_transform(subjects)

    # חילוץ תוצאות
    weights = tfidf_matrix.sum(axis=0).getA1()
    words = tfidf.get_feature_names_out()
    keywords_df = pd.DataFrame({'Word': words, 'Weight': weights}).sort_values(by='Weight', ascending=False)

    # הצגת גרף מילות המפתח
    plt.figure(figsize=(10, 8))
    sns.barplot(data=keywords_df, x='Weight', y='Word', palette='rocket')
    plt.title('Top Regulatory Keywords (TF-IDF Analysis)')
    plt.show()

    print("--- 10 המילים המובילות שמצאנו ---")
    print(keywords_df.head(10))

except Exception as e:
    print(f"חלה שגיאה: {e}")
    print("וודאי שהתקנת את הספרייה לקריאת אקסל על ידי הרצת: !pip install openpyxl")

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

# 1. טעינת הנתונים מהאקסל שכבר נמצא ב-Colab
df = pd.read_excel('/content/warning-letters.xlsx')

# 2. סינון ל-CDRH (מכשור רפואי) אם קייימת עמודה כזו
if 'Center' in df.columns:
    df_cdrh = df[df['Center'].str.contains('CDRH', na=False)].copy()
else:
    df_cdrh = df.copy()

print(f"עובדים על {len(df_cdrh)} מכתבים מהאקסל שלך.")

# פונקציה לחיפוש הטקסט המלא דרך גוגל/FDA בצורה ממוקדת
def get_full_text_by_meta(company, date):
    # כאן אנחנו בונים שאילתה שתחפש את המכתב הספציפי
    search_query = f"FDA Warning Letter {company} {date}"
    # מכיוון שחיפוש אוטומטי בגוגל הוא מורכב, בשלב זה של הפרויקט
    # אני מציע שנבצע ניתוח עומק על ה-'Subject' של כל 270 המכתבים
    # כדי לייצר את מילות המפתח (Keywords) שהן לב העבודה.
    return "Search logic needed"

# --- שלב ניתוח ה-Keywords העמוק מתוך ה-Subject ---
from sklearn.feature_extraction.text import TfidfVectorizer

# ניקוי ה-Subject
df_cdrh['Subject_Clean'] = df_cdrh['Subject'].fillna('').astype(str)

# שימוש ב-N-grams (צירופי מילים) כדי למצוא "חוסרים" (למשל 'failure to', 'lack of')
tfidf = TfidfVectorizer(stop_words='english', max_features=50, ngram_range=(1,3))
tfidf_matrix = tfidf.fit_transform(df_cdrh['Subject_Clean'])

# הצגת מילות המפתח המורכבות
words = tfidf.get_feature_names_out()
importance = tfidf_matrix.sum(axis=0).getA1()
keywords_df = pd.DataFrame({'Word': words, 'Importance': importance}).sort_values(by='Importance', ascending=False)

print("\n--- צירופי מילים משמעותיים שמצאנו (N-grams) ---")
print(keywords_df.head(15))

In [ ]:
# הסרת גרסאות ישנות והתקנה נקייה
!apt-get update
!apt-get install -y chromium-browser
!apt-get install -y chromium-chromedriver

# הגדרה של הנתיב לדרייבר
import sys
sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

print("התקנה הסתיימה בהצלחה!")

In [ ]:
# 1. התקנת הכלים הספציפיים ל-Colab (גרסת 2026)
!apt-get update
!apt install -y chromium-browser chromium-chromedriver
!pip install selenium

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import time
from tqdm import tqdm

# 2. הגדרת הפונקציה עם "תיקון הקריסה"
def fetch_full_text(company_name):
    if pd.isna(company_name): return "שם חסר"

    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')

    # הגדרת נתיב הדרייבר ב-Colab
    service = Service('/usr/bin/chromedriver')

    driver = webdriver.Chrome(service=service, options=options)
    try:
        # חיפוש ממוקד ב-FDA
        query = f"site:fda.gov warning letter {company_name}".replace(' ', '+')
        driver.get(f"https://www.google.com/search?q={query}")
        time.sleep(2)

        links = driver.find_elements(By.TAG_NAME, "a")
        target_url = None
        for link in links:
            url = link.get_attribute('href')
            if url and "fda.gov/inspections" in url:
                target_url = url
                break

        if target_url:
            driver.get(target_url)
            time.sleep(3)
            # חילוץ הטקסט המרכזי
            return driver.find_element(By.TAG_NAME, "main").text
        return "לא נמצא קישור"
    except:
        return "שגיאה טכנית בשליפה"
    finally:
        driver.quit()

# 3. הרצה על הנתונים שלך (עכשיו כשהעמודות ידועות)
try:
    # טעינה (כבר הוכחנו שזה אקסל)
    df = pd.read_excel('warning-letters (1).csv')

    # ניקוי ודגימה
    df_sample = df.head(3).copy()
    print(f"מתחיל שליפה עבור 3 חברות ראשונות...")

    results = []
    for name in tqdm(df_sample['Company Name']):
        results.append(fetch_full_text(name))

    df_sample['Full_Text'] = results

    print("\n--- הצלחה! הנה התוצאות שנשלפו: ---")
    print(df_sample[['Company Name', 'Full_Text']])

except Exception as e:
    print(f"שגיאה בהרצה: {e}")

In [ ]:
import urllib.request
from bs4 import BeautifulSoup

url = "https://www.fda.gov/inspections-compliance-enforcement-and-criminal-investigations/warning-letters/prorx-llc-723704-04072026"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
html = urllib.request.urlopen(req, timeout=30).read().decode("utf-8")

soup = BeautifulSoup(html, "html.parser")
main = soup.find("main") or soup.find("article")
text = main.get_text("\n", strip=True)
print(text)

In [ ]:
import urllib.request
import urllib.parse
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm

# הגדרת ה-Headers כדי למנוע חסימות (403 Forbidden)
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}

def get_warning_letters_links(category="Medical Service", max_pages=3):
    """
    שלב א': שולף את רשימת הקישורים למכתבי אזהרה מתוך עמודי האינדקס של ה-FDA
    """
    base_url = "https://www.fda.gov/inspections-compliance-enforcement-and-criminal-investigations/compliance-actions-and-activities/warning-letters"
    letters_data = []

    print(f"🔎 מחפש מכתבים בקטגוריה: {category}...")

    for page in range(0, max_pages):
        # בניית ה-URL עם הפרמטרים של הסינון (ה-FDA משתמש בפרמטרים אלו בדפי התוצאות שלהם)
        params = {
            'field_warning_letter_office_target_id': category, # או פרמטר הסינון המתאים של ה-FDA
            'page': page
        }
        url = f"{base_url}?{urllib.parse.urlencode(params)}"

        try:
            req = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(req, timeout=30) as response:
                soup = BeautifulSoup(response.read(), "html.parser")

            # מציאת הטבלה או רשימת המכתבים בדף (באתר ה-FDA לרוב מדובר בטבלה)
            table = soup.find('table')
            if not table:
                break

            links = table.find_all('a')
            page_added = 0

            for link in links:
                href = link.get('href', '')
                # וידאו שמדובר בקישור למכתב אזהרה
                if '/warning-letters/' in href:
                    full_url = urllib.parse.urljoin("https://www.fda.gov", href)
                    if full_url not in [l['url'] for l in letters_data]:
                        letters_data.append({
                            'title': link.get_text(strip=True),
                            'url': full_url
                        })
                        page_added += 1

            if page_added == 0: # אם הגענו לעמוד ריק מתוצאות חדשות
                break

        except Exception as e:
            print(f"שגיאה בסריקת עמוד אינדקס {page}: {e}")
            break

    print(f" Found {len(letters_data)} letters to download.")
    return pd.DataFrame(letters_data)

def fetch_letter_content(url):
    """
    שלב ב': הפונקציה המצוינת שלך - שולפת את התוכן של מכתב ספציפי
    """
    try:
        req = urllib.request.Request(url, headers=HEADERS)
        with urllib.request.urlopen(req, timeout=30) as response:
            html = response.read().decode("utf-8")

        soup = BeautifulSoup(html, "html.parser")
        main = soup.find("main") or soup.find("article") or soup.find("body")

        if main:
            return main.get_text("\n", strip=True)
        return "No content found"
    except Exception as e:
        return f"Error: {str(e)}"

# --- הרצה ראשית ---

# 1. שליפת רשימת הקישורים (ניתן להגדיל את max_pages לפי הצורך)
df_letters = get_warning_letters_links(category="Medical Service", max_pages=2)

if not df_letters.empty:
    # 2. מעבר על כל הקישורים ושליפת הטקסט המלא (עם מדד התקדמות ויזואלי)
    print("\n מושך את תוכן המכתבים (זה עשוי לקחת קצת זמן)...")
    tqdm.pandas()

    # הוספת השהייה קלה (sleep) כדי לא לחסום על ידי השרת של ה-FDA
    def safe_fetch(url):
        time.sleep(1) # שומר עליך מתחת לרדאר שלהם
        return fetch_letter_content(url)

    df_letters['Full_Text'] = df_letters['url'].progress_apply(safe_fetch)

    # 3. שמירה לקובץ אקסל/CSV
    df_letters.to_csv("fda_medical_service_letters.csv", index=False, encoding="utf-8-sig")
    print("\n הסתיים בהצלחה! הקובץ fda_medical_service_letters.csv מוכן.")
else:
    print("❌ לא נמצאו מכתבים בקטגוריה המבוקשת.")